# Set up

In [1]:
# install libraries
!pip install datasets
!pip install torch torchvision transformers

In [30]:
# Import libraries
from datasets import load_dataset, Dataset, DatasetDict
import pandas as pd
import torch
import torchvision
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import classification_report
import numpy as np

In [3]:
# check gpu is available
if torch.cuda.is_available():
  print("GPU is available")
else:
  print("GPU is not available")

GPU is available


In [4]:
# Import Go-Emotions
emotions_db = load_dataset("mrm8488/goemotions")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/7.11k [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


goemotions.csv: reconstructing file:   0%|          |  0.00B / 42.7MB            

goemotions.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/211225 [00:00<?, ? examples/s]

In [5]:
# Inspect dataset
emotions_db.set_format(type="pandas")


# Pre processing

In [11]:
# Audio2Face emotions
columns =  [
    'amazement', 'anger', 'cheekiness', 'disgust', 'fear', 'grief', 'joy', 'out of breath', 'pain', 'sadness', 'neutral'
]

# Custom dataframe
custom_df = pd.DataFrame(columns = columns)
train_db = emotions_db['train'].to_pandas()

# Emotion columns
custom_df['amazement'] = train_db['amusement'] + train_db['realization'] + train_db['surprise'] + train_db['admiration']
custom_df['anger'] = train_db['anger'] + train_db['annoyance'] + train_db['disapproval']
custom_df['cheekiness'] = train_db['caring'] + train_db['love'] + train_db['desire']
custom_df['disgust'] = train_db['disgust'] + train_db['remorse']
custom_df['fear'] = train_db['fear'] + train_db['nervousness']
custom_df['grief'] = train_db['grief']
custom_df['joy'] = train_db['joy'] + train_db['pride'] + train_db['optimism'] + train_db['gratitude'] + train_db['relief'] + train_db['excitement']
custom_df['out of breath'] = train_db['confusion']
custom_df['pain'] = train_db['disappointment']
custom_df['sadness'] = train_db['sadness'] + train_db['embarrassment']
custom_df['neutral'] = train_db['neutral'] + train_db['approval'] + train_db['curiosity']

# Text column
custom_df['text'] = train_db['text']

# Remove columns with all elements zero
custom_df = custom_df[(custom_df.T != 0).any()]
custom_dataset = Dataset.from_pandas(custom_df)

In [27]:
# Load tokenizer
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# Pre processing
def pre_process(examples):
  return tokenizer(examples['text'][0], truncation = True, padding = 'max_length')

# Apply tokenization with batches
distil_dataset = custom_dataset.map(pre_process, batch_size = 10)

Map:   0%|          | 0/211225 [00:00<?, ? examples/s]

# Format the dataset for the trainer

In [28]:
def add_label(example):
  example['label'] = [example[col] for col in columns]
  return example

distil_dataset = distil_dataset.map(add_label)

Map:   0%|          | 0/211225 [00:00<?, ? examples/s]

In [29]:
print(distil_dataset)

Dataset({
    features: ['amazement', 'anger', 'cheekiness', 'disgust', 'fear', 'grief', 'joy', 'out of breath', 'pain', 'sadness', 'neutral', 'text', 'input_ids', 'attention_mask', 'label'],
    num_rows: 211225
})


# Split into train, test and evaluation data

In [31]:
ds_train_devtest = distil_dataset.train_test_split(test_size=0.2, seed=42)
ds_devtest = ds_train_devtest['test'].train_test_split(test_size=0.5, seed=42)

ds_splits = DatasetDict({
    'train': ds_train_devtest['train'],
    'eval': ds_devtest['train'],
    'test': ds_devtest['test']
})
print(ds_splits)


DatasetDict({
    train: Dataset({
        features: ['amazement', 'anger', 'cheekiness', 'disgust', 'fear', 'grief', 'joy', 'out of breath', 'pain', 'sadness', 'neutral', 'text', 'input_ids', 'attention_mask', 'label'],
        num_rows: 168980
    })
    eval: Dataset({
        features: ['amazement', 'anger', 'cheekiness', 'disgust', 'fear', 'grief', 'joy', 'out of breath', 'pain', 'sadness', 'neutral', 'text', 'input_ids', 'attention_mask', 'label'],
        num_rows: 21122
    })
    test: Dataset({
        features: ['amazement', 'anger', 'cheekiness', 'disgust', 'fear', 'grief', 'joy', 'out of breath', 'pain', 'sadness', 'neutral', 'text', 'input_ids', 'attention_mask', 'label'],
        num_rows: 21123
    })
})


# Fine tuning

In [32]:
# Load DistilBERT model for classification
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=11)

# Setting up training settings
training_args = TrainingArguments(
    output_dir="./results",          # Directory for saving results
    learning_rate=5e-5,              # Initial learning rate
    per_device_train_batch_size=16,  # Batch size per GPU
    num_train_epochs=3,              # Number of epochs
    weight_decay=0.01,               # Regularization
    logging_dir="./logs",            # Directory for logs
    logging_steps=10                 # Log every 10 steps
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
trainer = Trainer(
    model = model,                             # The DistilBERT model
    args = training_args,                      # Training arguments
    train_dataset = ds_splits['train'],        # Training data
    eval_dataset= ds_splits['eval'],           # Validation data
)

# Start training
trainer.train()

Step,Training Loss
10,0.569000
20,0.397922
30,0.334234
40,0.303167
50,0.298197
60,0.291714
70,0.276334
80,0.305201
90,0.258610
100,0.274316


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

# Evaluation

In [ ]:
# Evaluate general performance
predictions = trainer.predict(ds_splits['test'])
preds = np.argmax(predictions.predictions, axis = 1)
actual_labels = ds_splits['test']['labels']
print(classification_report(actual_labels, preds))

# Error analysis
for idx, (actual, pred) in enumerate(zip(actual_labels, preds)):
  print(f"Example {idx}: \n")
  print(f"Sentence: {ds_splits['test']['text'][idx]} \n")
  print(f"Actual: { actual} \n")
  print(f"Predicted {predicted} \n")
  print("\n")


# Deployment

In [ ]:
# Save the model and tokenizer
model.save_pretrained("./fine_tuned_model")
tokenizer.save_pretrained("./fine_tuned_model")